In [ ]:
# Install required libraries
!pip install xgboost

In [ ]:
import numpy as np
import pandas as pd
import random

print("Libraries Imported Successfully!")

Libraries Imported Successfully!


In [ ]:
import pandas as pd
import numpy as np
import random

# Make results reproducible
random.seed(42)
np.random.seed(42)

data = []

# Generate 500 samples
for i in range(500):

    label = random.choice(["Planet", "False Positive"])

    if label == "Planet":
        depth = round(random.uniform(0.005, 0.03), 4)
        duration = round(random.uniform(1.5, 5), 2)
        period = round(random.uniform(2, 50), 2)
        snr = round(random.uniform(10, 30), 2)
        even_odd_ratio = round(random.uniform(0.95, 1.05), 2)
        secondary_eclipse = 0
        noise_level = round(random.uniform(0.01, 0.05), 3)
        transit_shape = "U"

    else:
        depth = round(random.uniform(0.03, 0.2), 4)
        duration = round(random.uniform(4, 12), 2)
        period = round(random.uniform(1, 30), 2)
        snr = round(random.uniform(2, 15), 2)
        even_odd_ratio = round(random.uniform(0.5, 0.9), 2)
        secondary_eclipse = random.choice([0,1])
        noise_level = round(random.uniform(0.03, 0.1), 3)
        transit_shape = random.choice(["V","Irregular","Smooth"])

    data.append([
        depth,
        duration,
        period,
        snr,
        even_odd_ratio,
        secondary_eclipse,
        noise_level,
        transit_shape,
        label
    ])

columns = [
    "Depth",
    "Duration",
    "Period",
    "SNR",
    "Even_Odd_Ratio",
    "Secondary_Eclipse",
    "Noise_Level",
    "Transit_Shape",
    "Label"
]

df = pd.DataFrame(data, columns=columns)

df.head()

,Depth,Duration,Period,SNR,Even_Odd_Ratio,Secondary_Eclipse,Noise_Level,Transit_Shape,Label
0,0.0056,2.46,12.71,24.73,1.02,0,0.046,U,Planet
1,0.0198,1.61,6.50,14.65,1.01,0,0.032,U,Planet
2,0.0675,8.71,24.47,2.08,0.82,1,0.054,V,False Positive
3,0.0289,2.68,6.45,11.93,1.03,0,0.034,U,Planet
4,0.0232,3.38,48.71,17.57,1.01,0,0.043,U,Planet


In [ ]:
df.to_csv("synthetic_dataset.csv", index=False)

print("Dataset Created Successfully!")
print("Total Samples:", len(df))

Dataset Created Successfully!
Total Samples: 500


In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("synthetic_dataset.csv")

df.head()

,Depth,Duration,Period,SNR,Even_Odd_Ratio,Secondary_Eclipse,Noise_Level,Transit_Shape,Label
0,0.0056,2.46,12.71,24.73,1.02,0,0.046,U,Planet
1,0.0198,1.61,6.50,14.65,1.01,0,0.032,U,Planet
2,0.0675,8.71,24.47,2.08,0.82,1,0.054,V,False Positive
3,0.0289,2.68,6.45,11.93,1.03,0,0.034,U,Planet
4,0.0232,3.38,48.71,17.57,1.01,0,0.043,U,Planet


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Create encoders
shape_encoder = LabelEncoder()
label_encoder = LabelEncoder()

# Convert text to numbers
df["Transit_Shape"] = shape_encoder.fit_transform(df["Transit_Shape"])
df["Label"] = label_encoder.fit_transform(df["Label"])

df.head()

,Depth,Duration,Period,SNR,Even_Odd_Ratio,Secondary_Eclipse,Noise_Level,Transit_Shape,Label
0,0.0056,2.46,12.71,24.73,1.02,0,0.046,2,1
1,0.0198,1.61,6.50,14.65,1.01,0,0.032,2,1
2,0.0675,8.71,24.47,2.08,0.82,1,0.054,3,0
3,0.0289,2.68,6.45,11.93,1.03,0,0.034,2,1
4,0.0232,3.38,48.71,17.57,1.01,0,0.043,2,1


In [ ]:
X = df.drop("Label", axis=1)

y = df["Label"]

print("Input Shape:", X.shape)
print("Output Shape:", y.shape)

Input Shape: (500, 8)
Output Shape: (500,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Training Samples: 400
Testing Samples: 100


In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100,
    random_state=42,
    eval_metric="logloss"
)

model.fit(X_train, y_train)

print("Model Trained Successfully!")

Model Trained Successfully!


In [ ]:
predictions = model.predict(X_test)

print(predictions[:20])

[1 1 1 0 0 1 0 0 1 1 0 0 0 0 0 1 1 0 1 1]


In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

print("Accuracy :", accuracy_score(y_test, predictions))
print("Precision:", precision_score(y_test, predictions))
print("Recall   :", recall_score(y_test, predictions))
print("F1 Score :", f1_score(y_test, predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

Confusion Matrix:
[[52  0]
 [ 0 48]]


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.utils import to_categorical

# Convert labels to categorical
y_train_cat = to_categorical(y_train)
y_test_cat = to_categorical(y_test)

# Reshape data for CNN
X_train_cnn = X_train.values.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_cnn = X_test.values.reshape((X_test.shape[0], X_test.shape[1], 1))

# Build CNN Model
cnn = Sequential()

cnn.add(Conv1D(32, kernel_size=2, activation='relu', input_shape=(X_train.shape[1],1)))
cnn.add(MaxPooling1D(pool_size=2))

cnn.add(Flatten())

cnn.add(Dense(32, activation='relu'))
cnn.add(Dense(2, activation='softmax'))

cnn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 7, 32)          │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 3, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 96)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,266 (12.76 KB)

 Trainable params: 3,266 (12.76 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = cnn.fit(
    X_train_cnn,
    y_train_cat,
    epochs=20,
    batch_size=16,
    validation_data=(X_test_cnn, y_test_cat)
)

Epoch 1/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.4875 - loss: 1.1431 - val_accuracy: 0.7300 - val_loss: 0.6237
Epoch 2/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8450 - loss: 0.5139 - val_accuracy: 0.8400 - val_loss: 0.4349
Epoch 3/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9250 - loss: 0.3437 - val_accuracy: 0.9200 - val_loss: 0.2746
Epoch 4/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9625 - loss: 0.2293 - val_accuracy: 0.9700 - val_loss: 0.2028
Epoch 5/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9800 - loss: 0.1707 - val_accuracy: 0.9600 - val_loss: 0.1484
Epoch 6/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9800 - loss: 0.1269 - val_accuracy: 0.9600 - val_loss: 0.1223
Epoch 7/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9825 - loss: 0.0991 - val_accuracy: 0.9700 - val_loss: 0.1090
Epoch 8/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9850 - loss: 0.0877 - val_accuracy: 0.9500 - val_l

In [ ]:
loss, accuracy = cnn.evaluate(X_test_cnn, y_test_cat)

print("CNN Accuracy:", accuracy)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9800 - loss: 0.0544
CNN Accuracy: 0.9800000190734863
